# 04번 진단: 이상치(액면분할/병합 의심) 제외 + 보유종목수 상한 없음(n=ALL) 비교

`check_04_01_backtest.ipynb`에서 `per8_pbr1.2_dynone_monthly_n10` 전략의 2023-09-30~10-31
구간이 종목 `101140` 하나의 +1900% 급등(액면분할/병합 의심)으로 통째로 왜곡됐다.

`04_backtest_grid.py` 코드 자체는 건드리지 않고(진단용), 이 노트북에서 직접:
1. 보유 기간 중 하루라도 `split_suspected=True`가 찍힌 종목은 그 구간 수익률 계산에서 제외
2. 보유종목수 상한을 없애고(PER/PBR 조건 만족 종목 전부) 계산

두 가지를 적용했을 때 원래 결과와 어떻게 달라지는지 비교한다.

**성능 참고**: 이 노트북은 구간마다 개별 액션(.first()/.count())을 반복하지 않고, 모든
변형(n10/nMAX)×구간(35개)의 계산을 하나의 지연(lazy) DataFrame으로 쌓아뒀다가 마지막에
`groupBy` 집계 한 번(`toPandas()` 호출 시점)으로 끝낸다 - 액션 수를 210여 개에서 1개로 줄여
체감 실행시간을 크게 단축했다.

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 이전 실행에서 toPandas() 단계가 java.lang.OutOfMemoryError(Java heap space)로 실패했음.
# 데이터 자체는 작지만(indicators 3천여행, held_all도 최대 1만행 안팎), n10/nMAX 2개 변형 x 35개 시점의
# screen_stocks() 서브쿼리(윈도우 함수 포함)가 전부 union되어 하나의 거대한 실행계획으로 쌓이면서,
# 드라이버 기본 힙(1GB)이 스테이지/태스크 메타데이터만으로도 부족해진 것으로 판단해 드라이버 메모리를 늘림.
spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_04_02") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

companies = spark.read.parquet("/opt/spark-apps/data/raw/companies").filter(F.col("corp_cls") == "Y")  # KOSPI
indicators = spark.read.parquet("/opt/spark-apps/data/indicators") \
    .join(F.broadcast(companies.select("stock_code")), on="stock_code", how="inner").cache()
prices = spark.read.parquet("/opt/spark-apps/data/cleaned/prices").filter(F.col("snapshot_type") == "current").cache()

print(f"KOSPI 종목수: {companies.count()}")
print(f"indicators 행수: {indicators.count()}")

KOSPI 종목수: 810
indicators 행수: 3094


## 1. 리밸런싱 시점 목록 생성 + 벡터화된 스크리닝/가격조회 함수

**주의**: 최초 버전은 35개 시점을 Python for문으로 돌며 `unionByName`을 반복 호출했는데,
매 반복마다 Catalyst가 그때까지 쌓인 전체 실행계획을 다시 분석하면서 Spark 잡이 하나도 안 뜬
채로(즉 계산 자체는 시작도 안 한 채) 몇 분씩 걸리는 문제가 실제로 발생했다. 아래는 그 대신
`crossJoin` + `Window.partitionBy(..., "rebalance_date")`로 모든 시점을 한 번에 처리하도록
재작성한 버전이다 (반복문 없음, union 없음).

In [12]:
from datetime import date, timedelta

def month_end(year, month):
    next_month_first = date(year + 1, 1, 1) if month == 12 else date(year, month + 1, 1)
    return (next_month_first - timedelta(days=1)).strftime("%Y%m%d")

def build_rebalance_dates(years, rebalance):
    months = range(1, 13) if rebalance == "monthly" else (3, 6, 9, 12)
    return sorted(month_end(y, m) for y in years for m in months)

## 2. 이상치(split_suspected) 날짜 목록
종목별로 split_suspected=True가 찍힌 날짜만 모아둔다 (실제 제외 판정은 3번 셀에서 구간 조인으로 처리).

In [13]:
split_flags = prices.filter(F.col("split_suspected") == True) \
    .select("stock_code", F.col("bas_dt").alias("flag_date")).cache()
print(f"split_suspected=True 로 찍힌 (종목,날짜) 행수: {split_flags.count()}")

split_suspected=True 로 찍힌 (종목,날짜) 행수: 1523


## 3. 벡터화된 스크리닝 + 가격조회 + 구간 수익률 계산 (반복문/union 없음)

`rebalance_dates`를 작은 Spark DataFrame으로 만들어 `indicators`/`prices`와 `crossJoin`하고,
`Window.partitionBy(..., "rebalance_date")`로 시점별 계산을 한 번에 벡터화한다.
실제 계산은 다음 셀의 `toPandas()` 한 번에 전부 실행된다.

In [14]:
PER_MAX, PBR_MAX, DY_MIN = 8, 1.2, None
years = [2021, 2022, 2023]
rebalance_dates = build_rebalance_dates(years, "monthly")
buy_dates = rebalance_dates[:-1]
sell_dates = rebalance_dates[1:]

# 1) 리밸런싱 시점 목록을 작은 Spark DataFrame으로 (크로스조인 상대편)
dates_df = spark.createDataFrame([(d,) for d in buy_dates], ["rebalance_date"]).cache()

# 2) 가격: KOSPI 종목만 먼저 좁힌 뒤 시점과 crossJoin (전체 시장 대상으로 하면 행수가 불필요하게 커짐)
prices_kospi = prices.join(F.broadcast(companies.select("stock_code")), on="stock_code", how="inner") \
    .select("stock_code", "bas_dt", "close_price")
all_dates_df = spark.createDataFrame([(d,) for d in rebalance_dates], ["rebalance_date"])
price_cross = prices_kospi.join(F.broadcast(all_dates_df), F.col("bas_dt") <= F.col("rebalance_date"), how="inner")
w_price = Window.partitionBy("stock_code", "rebalance_date").orderBy(F.desc("bas_dt"))
prices_by_date = price_cross.withColumn("_rank", F.row_number().over(w_price)).filter(F.col("_rank") == 1) \
    .select("stock_code", "rebalance_date", F.col("close_price").alias("price"))

# 3) 스크리닝: indicators x 시점을 한 번에 crossJoin 후, 시점별로 rcept_no<=시점 중 최신 것만 선택
ind_cross = indicators.join(F.broadcast(dates_df), F.col("rcept_no") <= F.col("rebalance_date"), how="inner")
w_latest = Window.partitionBy("stock_code", "rebalance_date").orderBy(F.desc("rcept_no"))
latest_per_date = ind_cross.withColumn("_rank", F.row_number().over(w_latest)).filter(F.col("_rank") == 1)

condition = (
    (F.col("per") > 0) & (F.col("per") <= PER_MAX)
    & (F.col("pbr") > 0) & (F.col("pbr") <= PBR_MAX)
    & (
        F.lit(DY_MIN).isNull()
        | (F.col("dividend_yield").isNotNull() & (F.col("dividend_yield") >= F.lit(DY_MIN)))
    )
)
eligible = latest_per_date.filter(condition)

w_rank = Window.partitionBy("rebalance_date").orderBy(F.col("per").asc())
ranked = eligible.withColumn("rank", F.row_number().over(w_rank))

n10_portfolio = ranked.filter(F.col("rank") <= 10).select("stock_code", "rebalance_date").withColumn("variant", F.lit("n10"))
nmax_portfolio = ranked.select("stock_code", "rebalance_date").withColumn("variant", F.lit("nMAX"))
portfolios_all = n10_portfolio.unionByName(nmax_portfolio)  # 딱 2개짜리 union - 반복문 아님

# 4) 구간(매수시점->매도시점) 매핑 테이블 + 매수/매도가 조인으로 종목별 수익률 벡터화 계산
periods_df = spark.createDataFrame(list(zip(buy_dates, sell_dates)), ["period_start", "period_end"])

port_with_period = portfolios_all.join(F.broadcast(periods_df), portfolios_all.rebalance_date == periods_df.period_start, "inner")

buy_prices = prices_by_date.select("stock_code", F.col("rebalance_date").alias("period_start"), F.col("price").alias("buy_price"))
sell_prices = prices_by_date.select("stock_code", F.col("rebalance_date").alias("period_end"), F.col("price").alias("sell_price"))

held_all = port_with_period.join(buy_prices, on=["stock_code", "period_start"], how="inner") \
    .join(sell_prices, on=["stock_code", "period_end"], how="inner") \
    .withColumn("stock_return", (F.col("sell_price") - F.col("buy_price")) / F.col("buy_price")) \
    .select("variant", "period_start", "period_end", "stock_code", "stock_return")

print("벡터화된 지연 DataFrame 구성 완료 (반복문/union 없음, 아직 계산 안 됨)")

벡터화된 지연 DataFrame 구성 완료 (반복문/union 없음, 아직 계산 안 됨)


## 4. 이상치 판정(구간 안에 flag_date가 있는가) + 최종 집계를 한 번의 액션으로 실행

In [15]:
# (stock_code, period_start, period_end) 조합 중 실제로 이상치인 것만 미리 중복제거(distinct)해서
# held_all과 다시 조인 - 중복 조인으로 인한 행 뻥튀기 방지
outlier_keys = held_all.select("stock_code", "period_start", "period_end").distinct().alias("h") \
    .join(
        split_flags.alias("f"),
        (F.col("h.stock_code") == F.col("f.stock_code"))
        & (F.col("f.flag_date") > F.col("h.period_start"))
        & (F.col("f.flag_date") <= F.col("h.period_end")),
        how="inner",
    ).select("h.stock_code", "h.period_start", "h.period_end").distinct() \
    .withColumn("is_outlier", F.lit(True))

with_outlier_flag = held_all.join(outlier_keys, on=["stock_code", "period_start", "period_end"], how="left") \
    .fillna({"is_outlier": False})

grouped = with_outlier_flag.groupBy("variant", "period_start", "period_end").agg(
    F.avg("stock_return").alias("raw_return"),
    F.count("*").alias("raw_held"),
    F.avg(F.when(~F.col("is_outlier"), F.col("stock_return"))).alias("filtered_return"),
    F.sum(F.when(~F.col("is_outlier"), 1).otherwise(0)).alias("filtered_held"),
    F.sum(F.when(F.col("is_outlier"), 1).otherwise(0)).alias("outlier_excluded"),
).orderBy("variant", "period_start")

final_pdf = grouped.toPandas()  # 이 한 줄에서만 실제 계산이 실행됨 (단일 액션)
print(f"결과 행수: {len(final_pdf)}")
final_pdf.head()

Py4JJavaError: An error occurred while calling o765.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 16.0 failed 4 times, most recent failure: Lost task 2.3 in stage 16.0 (TID 47) (172.19.0.7 executor 0): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 8) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(Pyth

## 5. n10 결과 비교 (이상치 포함 vs 제외) - 101140 케이스가 실제로 걸러지는지 확인

In [ ]:
n10_df = final_pdf[final_pdf["variant"] == "n10"].reset_index(drop=True)
n10_df["raw_return_pct"] = n10_df["raw_return"] * 100
n10_df["filtered_return_pct"] = n10_df["filtered_return"] * 100
n10_df

## 6. nMAX(보유종목수 상한 없음) 결과

In [ ]:
nmax_df = final_pdf[final_pdf["variant"] == "nMAX"].reset_index(drop=True)
nmax_df["raw_return_pct"] = nmax_df["raw_return"] * 100
nmax_df["filtered_return_pct"] = nmax_df["filtered_return"] * 100
nmax_df

## 7. 누적수익률 곡선 - 4가지 비교 (n10 raw/filtered, nMAX raw/filtered)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
for variant_name, df in (("n10", n10_df), ("nMAX", nmax_df)):
    df["cum_raw"] = (1 + df["raw_return"]).cumprod() - 1
    df["cum_filtered"] = (1 + df["filtered_return"]).cumprod() - 1
    ax.plot(df["period_end"], df["cum_raw"], label=f"{variant_name} (이상치 포함)", linestyle="--")
    ax.plot(df["period_end"], df["cum_filtered"], label=f"{variant_name} (이상치 제외)")

ax.set_xticks(range(0, len(n10_df), 3))
ax.set_xticklabels(n10_df["period_end"].iloc[::3], rotation=90)
ax.set_title("PER8/PBR1.2 monthly - 이상치 포함/제외 x 상한 있음/없음 비교")
ax.set_ylabel("누적수익률")
ax.legend()
plt.tight_layout()
plt.show()

## 8. 최종 요약: 4가지 조합의 최종 누적수익률/총 이상치 제외 건수

In [ ]:
import pandas as pd

summary_rows = []
for variant_name, df in (("n10", n10_df), ("nMAX", nmax_df)):
    summary_rows.append({
        "variant": variant_name,
        "final_cum_raw": df["cum_raw"].iloc[-1],
        "final_cum_filtered": df["cum_filtered"].iloc[-1],
        "total_outliers_excluded": df["outlier_excluded"].sum(),
    })
pd.DataFrame(summary_rows)

In [10]:
spark.stop()